In [49]:
import numpy as np
import pandas as pd
import NeuralNet
import layer

In [50]:
datasetNames = ['ID number', 'Diagnosis', 
                'Mean Radius', 'Mean Texture', 'Mean Perimeter', 'Mean Area', 'Mean Smoothness', 'Mean Compactness', 'Mean Concavity', 'Mean Concave Points', 'Mean Symmetry', 'Mean Fractal Dimension',
                'SE Radius', 'SE Texture', 'SE Perimeter', 'SE Area', 'SE Smoothness', 'SE Compactness', 'SE Concavity', 'SE Concave Points', 'SE Symmetry', 'SE Fractal Dimension',
                'Worst Radius', 'Worst Texture', 'Worst Perimeter', 'Worst Area', 'Worst Smoothness', 'Worst Compactness', 'Worst Concavity', 'Worst Concave Points', 'Worst Symmetry', 'Worst Fractal Dimension']
dataset = pd.read_csv('data/wdbc.data', names = datasetNames, sep = ',')
dataset = dataset.drop(columns=['ID number'])
dataset.head()

,Diagnosis,Mean Radius,Mean Texture,Mean Perimeter,Mean Area,Mean Smoothness,Mean Compactness,Mean Concavity,Mean Concave Points,Mean Symmetry,...,Worst Radius,Worst Texture,Worst Perimeter,Worst Area,Worst Smoothness,Worst Compactness,Worst Concavity,Worst Concave Points,Worst Symmetry,Worst Fractal Dimension
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [51]:
dataset = dataset.replace({'M':0,'B':1})                                                     
dataset.head()

,Diagnosis,Mean Radius,Mean Texture,Mean Perimeter,Mean Area,Mean Smoothness,Mean Compactness,Mean Concavity,Mean Concave Points,Mean Symmetry,...,Worst Radius,Worst Texture,Worst Perimeter,Worst Area,Worst Smoothness,Worst Compactness,Worst Concavity,Worst Concave Points,Worst Symmetry,Worst Fractal Dimension
0,0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,0,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,0,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,0,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,0,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [52]:
def kfolds(dataset, nfolds):
    folds = []
    foldLength = int(dataset.shape[0]/nfolds)

    dataset = dataset.reindex(np.random.permutation(dataset.index))                                                          
    dataset = dataset.reset_index(drop = True)
    
    idx = 0
    lidx = foldLength - 1
    for i in range(nfolds):
        folds.append(dataset.loc[idx : lidx])
        idx += foldLength
        lidx += foldLength
    return folds

In [53]:
def kfoldsplit(folds):
    train_test = []

    for i in range(len(folds)):
        temp = []
        folds_copy = folds.copy()

        test = folds_copy.pop(i)
        test = test.drop(test.columns[0], axis = 1)

        temp.append(pd.concat(folds_copy))
        temp.append(test)
        
        train_test.append(temp)
    return train_test

In [54]:
def kfoldcv(dataset, nfolds = 5):
    folds = kfolds(dataset, nfolds)
    train_test = kfoldsplit(folds)

    accuracies = []
    for i in train_test:
        X_test = np.asarray(i[-1])
        X_train = np.array(i[0].drop(i[0].columns[0], axis = 1))
        y_train = np.array(i[0].iloc[:,0])

        ann = NeuralNet.ANN(0.1, 2, X_train.T, y_train.T)
        activations = [2,2]
        ann.setLayers(activations, 2, 1)

        ann.train_sgd()

        accuracy = ann.test(X_test)
        accuracies.append(accuracy)
    
    return accuracies
kfoldcv(dataset, 4)

the activations are [2, 2]
the variable argument is  (2, 1)
number of hidden layers 1 

input to the network is [[2.309e+01 9.676e+00 1.953e+01 ... 2.201e+01 1.719e+01 1.263e+01]
 [1.983e+01 1.314e+01 1.890e+01 ... 2.190e+01 2.207e+01 2.076e+01]
 [1.521e+02 6.412e+01 1.295e+02 ... 1.472e+02 1.116e+02 8.215e+01]
 ...
 [2.264e-01 1.075e-01 1.980e-01 ... 2.432e-01 1.984e-01 1.105e-01]
 [2.908e-01 2.848e-01 2.968e-01 ... 2.741e-01 3.216e-01 2.226e-01]
 [7.277e-02 1.364e-01 9.929e-02 ... 8.574e-02 7.570e-02 8.486e-02]]
the output is [0 1 0 1 1 0 0 0 0 0 1 0 1 0 1 0 0 1 1 0 1 0 1 1 1 1 1 1 0 1 0 1 0 1 1 1 0
 1 0 1 1 1 1 1 1 0 0 1 0 1 1 1 0 0 0 1 1 0 1 0 0 0 1 1 1 0 1 1 0 1 1 1 0 0
 0 1 0 0 1 0 1 1 1 0 1 0 0 0 0 1 0 0 1 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1
 1 0 0 0 1 0 1 1 1 1 1 0 0 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 0 0 1
 0 0 1 1 1 1 1 0 0 1 1 1 0 1 0 1 1 0 0 1 1 0 0 1 1 1 1 1 1 0 0 1 0 1 0 0 1
 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 0 0 1 1 1 1 0 1 0 1 1 1 1 0 1 1 1 1
 1 1 1 1 1 1 1 1

AttributeError: 'ANN' object has no attribute 'test'